# Critic Networks

> Specific models that can serve as critic networks for RL agents

In [ ]:
#| default_exp RL_approximators

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

# import logging
# logging_level = logging.DEBUG

from abc import ABC, abstractmethod
from typing import Union, Tuple, List
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import trange, tqdm

from mushroom_rl.core import Serializable
from mushroom_rl.utils.minibatches import minibatch_generator
from mushroom_rl.utils.torch import get_weights, set_weights, zero_grad, update_optimizer_parameters
import time

In [ ]:
# | export
class RNNWrapper(nn.Module):
    def __init__(self, rnn_cell_class, *args, **kwargs):
        """
        Initializes the RNNWrapper with the specified RNN cell.
        
        Parameters:
        - rnn_cell_class: The RNN cell class (e.g., nn.GRU, nn.LSTM, nn.RNN).
        - *args, **kwargs: The arguments and keyword arguments to be passed to the RNN cell.
        """
        super(RNNWrapper, self).__init__()
        self.rnn = rnn_cell_class(*args, **kwargs)

    def forward(self, x):
        output, _ = self.rnn(x)  # Extract and return only the output
        return output

    @classmethod
    def create(cls, rnn_cell_class):
        """
        A factory method to create a new RNNWrapper subclass with a specific RNN cell.
        
        Parameters:
        - rnn_cell_class: The RNN cell class to be wrapped (e.g., nn.GRU, nn.LSTM).
        
        Returns:
        - A new subclass of RNNWrapper.
        """
        class SpecificRNNWrapper(cls):
            def __init__(self, *args, **kwargs):
                super(SpecificRNNWrapper, self).__init__(rnn_cell_class, *args, **kwargs)

        return SpecificRNNWrapper

In [ ]:
# | export
class RNNWrapperHS(nn.Module):
    def __init__(self, rnn_cell_class, *args, **kwargs):
        """
        Initializes the RNNWrapper with the specified RNN cell.
        """
        super(RNNWrapperHS, self).__init__()
        self.rnn = rnn_cell_class(*args, **kwargs)

    def forward(self, x, hidden_state=None):
        """
        Forward pass of the RNN.

        Args:
            x: Input tensor of shape (batch, seq_len, input_dim)
            hidden_state: Initial hidden state (optional).

        Returns:
            output: Output features from the last layer of the RNN for each timestep
            hidden_state: Hidden state for the next timestep
        """
           
        output, hidden_state = self.rnn(x, hidden_state)
        return output, hidden_state

    @classmethod
    def create(cls, rnn_cell_class):
        """
        Factory method to create a new RNNWrapper subclass.
        """
        class SpecificRNNWrapperHS(cls):
            def __init__(self, *args, **kwargs):
                super(SpecificRNNWrapperHS, self).__init__(rnn_cell_class, *args, **kwargs)

        return SpecificRNNWrapperHS


In [ ]:
#| export
class BaseApproximator():

    """ Some basic functions for approximators """

    def __init__(self):
        super().__init__()

    def init_weights(self, layer, init_method, activation):
        """ Initialize the weights of a layer """
        init_method_function = self.select_init_method(init_method)
        
        # Check if initialization method requires gain
        if init_method in ["xavier_uniform", "xavier_normal"]:
            if activation == "identity":
                activation = "linear"
            gain = nn.init.calculate_gain(activation)
            init_method_function(layer.weight, gain=gain)
        else:
            init_method_function(layer.weight)
            
    def init_rnn_weights(self, rnn_layer, init_method):
        """Initialize the weights for the RNN layer."""
        init_method_function = self.select_init_method(init_method)

        for name, param in rnn_layer.named_parameters():
            if 'weight' in name:
                # Initialize weights using the selected method
                init_method_function(param)
            elif 'bias' in name:
                # Initialize biases to zero
                nn.init.constant_(param, 0)
                
    @staticmethod
    def select_rnn_cell(RNN_cell, return_hidden_state=False):
        """ Select the RNN cell based on input string """
        RNN_cell = RNN_cell.lower()  # Convert input to lowercase for consistency
        if RNN_cell == "gru":
            rnn_cell_class = nn.GRU
        elif RNN_cell == "lstm":
            rnn_cell_class = nn.LSTM
        elif RNN_cell == "rnn":
            rnn_cell_class = nn.RNN
        else:
            raise ValueError(f"RNN cell '{RNN_cell}' not recognized")
        if return_hidden_state:
            return RNNWrapperHS.create(rnn_cell_class)
        return RNNWrapper.create(rnn_cell_class)

    @staticmethod
    def select_init_method(init_method):
        """ Select the initialization method based on input string """
        init_method = init_method.lower()
        if init_method in ["xavier_uniform", "xavier"]:
            return nn.init.xavier_uniform_
        elif init_method in ["xavier_normal", "xaviernorm"]:
            return nn.init.xavier_normal_
        elif init_method in ["he_uniform", "kaiming_uniform"]:
            return nn.init.kaiming_uniform_
        elif init_method in ["he_normal", "kaiming_normal"]:
            return nn.init.kaiming_normal_
        elif init_method in ["normal", "gaussian"]:
            return nn.init.normal_
        elif init_method == "uniform":
            return nn.init.uniform_
        else:
            raise ValueError("Initialization method not recognized")

    @staticmethod
    def select_activation(activation):
        """ Select the activation function based on input string """
        activation = activation.lower()  # Convert input to lowercase for consistency
        if activation == "relu":
            return nn.ReLU
        elif activation == "sigmoid":
            return nn.Sigmoid
        elif activation == "tanh":
            return nn.Tanh
        elif activation == "elu":
            return nn.ELU
        elif activation == "leakyrelu":
            return nn.LeakyReLU
        elif activation == "identity":
            return nn.Identity
        else:
            raise ValueError(f"Activation function {activation} not recognized")

    def forward(self, x):
        """ Forward pass through the network - overwrite this if necessary """
        return self.model(x)

In [ ]:
#| export
class BaseApproximatorMLP(BaseApproximator, nn.Module):

    """ Some basic functions for approximators """

    def __init__(self):
        super().__init__()
    
    def build_MLP(  self,
                    input_size: int,
                    output_size: int,
                    hidden_layers: list,
                    activation: str = "relu",
                    drop_prob: float = 0.0,
                    batch_norm: bool = False,
                    final_activation: str = "identity",
                    init_method: str = "xavier_uniform" # Parameter for initialization
                  ):

        """ Builds a multi-layer perceptron (MLP) """

        HiddenActivation = self.select_activation(activation)
        FinalActivation = self.select_activation(final_activation)

        layers = []

        # Hidden layers
        last_size = input_size
        for num_neurons in hidden_layers:
            
            linear_layer = nn.Linear(last_size, num_neurons)
            self.init_weights(linear_layer, init_method, activation)
            layers.append(linear_layer)

            if batch_norm:
                layers.append(nn.BatchNorm1d(num_neurons))
        
            layers.append(HiddenActivation())
            layers.append(nn.Dropout(p=drop_prob))

            last_size = num_neurons
        
        # Output layer
        output_layer = nn.Linear(last_size, output_size)
        self.init_weights(output_layer, init_method, final_activation)
        layers.append(output_layer)
        layers.append(FinalActivation())

        # Combine layers
        model = nn.Sequential(*layers)

        self.model = model

In [ ]:
#| export
class RNNMLPHybrid(nn.Module, BaseApproximator):

    """ A hybrid model combining an RNN and an MLP """

    def __init__(self, 
                 RNN_input_size: int,
                 MLP_input_size: int | None,
                 output_size: int,
                 num_hidden_units_RNN: int,
                 hidden_layers_RNN: int,
                 hidden_layers_MLP: List[int],
                 hidden_layers_input_MLP: List[int] | None,
                 RNN_cell: nn.Module,
                 activation: nn.Module,
                 final_activation: nn.Module,
                 drop_prob: float,
                 batch_norm: bool,
                 init_method: str):
        super(RNNMLPHybrid, self).__init__()

        HiddenActivation = self.select_activation(activation)
        FinalActivation = self.select_activation(final_activation)
        RNNCell = self.select_rnn_cell(RNN_cell)

        # RNN
        # RNN layers

        rnn_layers = []
        rnn = RNNCell(RNN_input_size, num_hidden_units_RNN, hidden_layers_RNN, batch_first=True, dropout=drop_prob)
        self.init_rnn_weights(rnn, init_method)
        hidden_activation_rnn = HiddenActivation()  # Activation used after RNN layers
        rnn_layers.append(rnn)
        rnn_layers.append(hidden_activation_rnn)

        self.rnn = nn.Sequential(*rnn_layers)

        # Input MLP, if required:
        last_size = 0 if MLP_input_size is None else MLP_input_size
        if hidden_layers_input_MLP is not None:

            if last_size == 0:
                raise ValueError("MLP input size must be specified if input MLP is used")
            
            layers_input_MLP = []
            for num_neurons in hidden_layers_input_MLP:
                linear_layer = nn.Linear(last_size, num_neurons)
                self.init_weights(linear_layer, init_method, activation)
                layers_input_MLP.append(linear_layer)
                if batch_norm:
                    layers_input_MLP.append(nn.BatchNorm1d(num_neurons))
                layers_input_MLP.append(HiddenActivation())
                layers_input_MLP.append(nn.Dropout(p=drop_prob))
                last_size = num_neurons
            
            self.input_mlp = nn.Sequential(*layers_input_MLP)
        else:
            self.input_mlp = None
        
        # Main MLP layers
        layers_MLP = []
        last_size = num_hidden_units_RNN + last_size
        for num_neurons in hidden_layers_MLP:
            linear_layer = nn.Linear(last_size, num_neurons)
            self.init_weights(linear_layer, init_method, activation)
            layers_MLP.append(linear_layer)
            if batch_norm:
                layers_MLP.append(nn.BatchNorm1d(num_neurons))
            layers_MLP.append(HiddenActivation())
            layers_MLP.append(nn.Dropout(p=drop_prob))
            last_size = num_neurons

        # Output layer
        output_layer = nn.Linear(last_size, output_size)
        self.init_weights(output_layer, init_method, final_activation)
        layers_MLP.append(output_layer)

        self.main_mlp = nn.Sequential(*layers_MLP)

    
    def forward(self, x_rnn, x_mlp=None):
        # RNN

        rnn_out = self.rnn(x_rnn) # Only one output due to the wrapper
        rnn_out = rnn_out[:, -1, :]  # Take the last output of the RNN
        
        # Input MLP
        if x_mlp is not None:
            if self.input_mlp is not  None:
                x_mlp = self.input_mlp(x_mlp)
            x = torch.cat((rnn_out, x_mlp), dim=1)
        else:
            x = rnn_out

        # Main MLP
        x = self.main_mlp(x)

        return x

In [ ]:
# | export
class TorchApproximator(Serializable):
    """
    Revised TorchApproximator to support meta-episodes and explicit hidden state management.
    This version handles recurrent networks that return a tuple (prediction, hidden_state)
    and optionally accepts an initial hidden state via kwargs (e.g., init_hidden) for meta-episode unrolling.
    """
    def __init__(self, input_shape, output_shape, network, optimizer=None,
                 loss=None, batch_size=0, n_fit_targets=1, use_cuda=False,
                 reinitialize=False, dropout=False, quiet=True, **params):
        """
        Constructor.
        
        Args:
            input_shape (tuple): shape of the input of the network;
            output_shape (tuple): shape of the output of the network;
            network (torch.nn.Module): the network class to use;
            optimizer (dict): the optimizer used for every fit step;
            loss (torch.nn.functional): the loss function to optimize in the
                fit method;
            batch_size (int, 0): the size of each minibatch. If 0, the whole
                dataset is fed to the optimizer at each epoch;
            n_fit_targets (int, 1): the number of fit targets used by the fit
                method of the network;
            use_cuda (bool, False): if True, runs the network on the GPU;
            reinitialize (bool, False): if True, the approximator is re
                initialized at every fit call. To perform the initialization, 
                the weights_init method must be defined properly for the 
                selected model network.
            dropout (bool, False): if True, dropout is applied only during
                train;
            quiet (bool, True): if False, shows two progress bars, one for
                epochs and one for the minibatches;
            **params: dictionary of parameters needed to construct the
                network.

        """
        self._batch_size = batch_size
        self._reinitialize = reinitialize
        self._use_cuda = use_cuda
        self._dropout = dropout
        self._quiet = quiet
        self._n_fit_targets = n_fit_targets

        # Initialize network.
        self.network = network(input_shape, output_shape, use_cuda=use_cuda,
                               dropout=dropout, **params)
        # New change: set a flag if the network supports hidden state (i.e., has an `init_hidden` method).
        self.supports_hidden_state = hasattr(self.network, 'init_hidden')

        if self._use_cuda:
            self.network.cuda()
        if self._dropout:
            self.network.eval()

        if optimizer is not None:
            self._optimizer = optimizer['class'](self.network.parameters(),
                                                 **optimizer['params'])
        self._loss = loss

        self._add_save_attr(
            _batch_size='primitive',
            _reinitialize='primitive',
            _use_cuda='primitive',
            _dropout='primitive',
            _quiet='primitive',
            _n_fit_targets='primitive',
            network='torch',
            _optimizer='torch',
            _loss='pickle',
            _last_loss='none'
        )

        self._last_loss = None

    def predict(self, *args, output_tensor=False, **kwargs):
        """
        Predict.

        Args:
            *args: input;
            output_tensor (bool, False): whether to return the output as tensor
                or not;
            **kwargs: other parameters used by the predict method
                the regressor.

        Returns:
            The predictions of the model.

        """
        if not self._use_cuda:
            torch_args = [torch.as_tensor(x) if isinstance(x, np.ndarray) else x
                          for x in args]
            val = self.network(*torch_args, **kwargs)

            if output_tensor:
                return val
            elif isinstance(val, tuple):
                val = tuple([x.detach().numpy() for x in val])
            else:
                val = val.detach().numpy()
        else:
            torch_args = [torch.as_tensor(x).cuda()
                          if isinstance(x, np.ndarray) else x.cuda() for x in args]
            val = self.network(*torch_args, **kwargs)

            if output_tensor:
                return val
            elif isinstance(val, tuple):
                val = tuple([x.detach().cpu().numpy() for x in val])
            else:
                val = val.detach().cpu().numpy()

        return val

    def fit(self, *args, n_epochs=None, weights=None, epsilon=None, patience=1,
            validation_split=1., **kwargs):
        """
        Fit the model.

        Args:
            *args: input, where the last ``n_fit_targets`` elements
                are considered as the target, while the others are considered
                as input;
            n_epochs (int, None): the number of training epochs;
            weights (np.ndarray, None): the weights of each sample in the
                computation of the loss;
            epsilon (float, None): the coefficient used for early stopping;
            patience (float, 1.): the number of epochs to wait until stop
                the learning if not improving;
            validation_split (float, 1.): the percentage of the dataset to use
                as training set;
            **kwargs: other parameters used by the fit method of the
                regressor.
                
        New changes:
        - The kwargs can include a flag (e.g., meta_episode=True) and an
          optional 'init_hidden' to support meta-episode unrolling.
        """
        if self._reinitialize:
            self.network.weights_init()

        if self._dropout:
            self.network.train()

        if epsilon is not None:
            n_epochs = np.inf if n_epochs is None else n_epochs
            check_loss = True
        else:
            n_epochs = 1 if n_epochs is None else n_epochs
            check_loss = False

        if weights is not None:
            args += (weights,)
            use_weights = True
        else:
            use_weights = False

        if 0 < validation_split <= 1:
            train_len = np.ceil(len(args[0]) * validation_split).astype(int)
            train_args = [a[:train_len] for a in args]
            val_args = [a[train_len:] for a in args]
        else:
            raise ValueError("Invalid validation split.")

        patience_count = 0
        best_loss = np.inf
        epochs_count = 0
        if check_loss:
            with tqdm(total=n_epochs if n_epochs < np.inf else None,
                      dynamic_ncols=True, disable=self._quiet,
                      leave=False) as t_epochs:
                while patience_count < patience and epochs_count < n_epochs:
                    mean_loss_current = self._fit_epoch(train_args, use_weights, kwargs)
                    if len(val_args[0]):
                        mean_val_loss_current = self._compute_batch_loss(val_args, use_weights, kwargs)
                        loss_val = mean_val_loss_current.item()
                    else:
                        loss_val = mean_loss_current

                    if not self._quiet:
                        t_epochs.set_postfix(loss=loss_val)
                        t_epochs.update(1)

                    if best_loss - loss_val > epsilon:
                        patience_count = 0
                        best_loss = loss_val
                    else:
                        patience_count += 1

                    self._last_loss = mean_loss_current
                    epochs_count += 1
        else:
            with trange(n_epochs, disable=self._quiet) as t_epochs:
                for _ in t_epochs:
                    mean_loss_current = self._fit_epoch(train_args, use_weights, kwargs)
                    if not self._quiet:
                        t_epochs.set_postfix(loss=mean_loss_current)
                    self._last_loss = mean_loss_current

        if self._dropout:
            self.network.eval()

    def _fit_epoch(self, args, use_weights, kwargs):
        # New change: allow minibatch generation to work with meta-episodes.
        if self._batch_size > 0:
            batches = minibatch_generator(self._batch_size, *args)
        else:
            batches = [args]

        loss_current = []
        for batch in batches:
            loss_current.append(self._fit_batch(batch, use_weights, kwargs))
        return np.mean(loss_current)

    def _fit_batch(self, batch, use_weights, kwargs):
        loss = self._compute_batch_loss(batch, use_weights, kwargs)
        self._optimizer.zero_grad()
        loss.backward()
        self._optimizer.step()
        return loss.item()

    def _compute_batch_loss(self, batch, use_weights, kwargs):
        """
        New change:
        - If using weights, they are processed as before.
        - The network forward pass now checks for tuple returns (prediction, hidden_state).
        - Additionally, if kwargs contains 'meta_episode' flag, you could adjust the processing
          (e.g., by not flattening the time dimension). For simplicity, here we assume that in meta_episode
          mode the network already handles sequences.
        """
        if use_weights:
            weights = torch.as_tensor(batch[-1]).type(torch.float)
            if self._use_cuda:
                weights = weights.cuda()
            batch = batch[:-1]

        # Convert inputs to tensors on the proper device.
        if not self._use_cuda:
            torch_args = [torch.as_tensor(x) for x in batch]
        else:
            torch_args = [torch.as_tensor(x).cuda() for x in batch]

        # Inputs (features) are all arguments except the last n_fit_targets, which are the targets.
        x = torch_args[:-self._n_fit_targets]

        # Optional: Pass along recurrent state info if supplied (e.g., 'init_hidden')
        # The 'init_hidden' in kwargs should be prepared externally if needed.
        y_hat = self.network(*x, **kwargs)
        # New change: if network output is a tuple, extract only the prediction tensor.
        if isinstance(y_hat, tuple):
            # You might want to log or handle the hidden state if needed
            _hidden_state = y_hat[1]  # hidden state is obtained but not used in loss computation
            y_hat = y_hat[0]

        # Determine the type for target tensors.
        if isinstance(y_hat, tuple):
            output_type = y_hat[0].dtype
        else:
            output_type = y_hat.dtype

        y = [y_i.clone().detach().type(output_type) for y_i in torch_args[-self._n_fit_targets:]]

        if self._use_cuda:
            y = [y_i.cuda() for y_i in y]

        if not use_weights:
            loss = self._loss(y_hat, *y)
        else:
            loss = self._loss(y_hat, *y, reduction='none')
            loss @= weights
            loss = loss / weights.sum()
        return loss

    def set_weights(self, weights):
        """
        Setter.
        """
        set_weights(self.network.parameters(), weights, self._use_cuda)

    def get_weights(self):
        """
        Getter.
        """
        return get_weights(self.network.parameters())

    @property
    def weights_size(self):
        """
        Returns:
            The size of the array of weights.
        """
        return sum(p.numel() for p in self.network.parameters())

    def diff(self, *args, **kwargs):
        """
        Compute the derivative of the output w.r.t. input.
        """
        if not self._use_cuda:
            torch_args = [torch.as_tensor(np.atleast_2d(x)) for x in args]
        else:
            torch_args = [torch.as_tensor(np.atleast_2d(x)).cuda() for x in args]

        y_hat = self.network(*torch_args, **kwargs)
        # New change: if y_hat is a tuple, use only the prediction.
        if isinstance(y_hat, tuple):
            y_hat = y_hat[0]
        n_outs = 1 if len(y_hat.shape) == 0 else y_hat.shape[-1]
        y_hat = y_hat.view(-1, n_outs)

        gradients = []
        for i in range(y_hat.shape[1]):
            zero_grad(self.network.parameters())
            y_hat[:, i].backward(retain_graph=True)
            grad_list = []
            for p in self.network.parameters():
                g = p.grad.data.detach().cpu().numpy()
                grad_list.append(g.flatten())
            gradients.append(np.concatenate(grad_list, 0))
        return np.stack(gradients, -1)

    @property
    def use_cuda(self):
        return self._use_cuda

    @property
    def loss_fit(self):
        """
        Returns:
            The average loss of the last epoch of the last fit call.
        """
        return self._last_loss

    def _post_load(self):
        if self._optimizer is not None:
            update_optimizer_parameters(self._optimizer, list(self.network.parameters()))


In [ ]:
# | export
class RL2RNN(nn.Module, BaseApproximator):
    """
    Pure RL² recurrent neural network module.
    Inherits BaseApproximator for initialization and activation utilities.
    """

    def __init__(self, RNN_input_size: int,
                 output_size: int,
                 hidden_layers_RNN: int,
                 num_hidden_units_RNN: int,
                 hidden_layers_MLP: List[int], 
                 RNN_cell: str,
                 activation: str,
                 final_activation: str,
                 drop_prob: float,
                 batch_norm: bool,
                 init_method: str):
        super(RL2RNN, self).__init__()
        
        HiddenActivation = self.select_activation(activation)
        FinalActivation = self.select_activation(final_activation)
        RNNCell = self.select_rnn_cell(RNN_cell, return_hidden_state=True)

        self.hidden_dim = num_hidden_units_RNN
        self.rnn_cell_type = RNN_cell.lower()

        # RNN Core (single GRU or LSTM)
        self.rnn_core = RNNCell(
            input_size=RNN_input_size,
            hidden_size=num_hidden_units_RNN,
            num_layers=hidden_layers_RNN,
            batch_first=True,
            dropout=drop_prob
        )
        self.init_rnn_weights(self.rnn_core, init_method)

        # MLP Head
        layers = []
        last_size = num_hidden_units_RNN
        for size in hidden_layers_MLP:
            linear = nn.Linear(last_size, size)
            self.init_weights(linear, init_method, activation)
            layers.append(linear)
            if batch_norm:
                layers.append(nn.BatchNorm1d(size))
            layers.append(HiddenActivation())
            layers.append(nn.Dropout(drop_prob))
            last_size = size

        # Final output layer
        output_layer = nn.Linear(last_size, output_size)
        self.init_weights(output_layer, init_method, final_activation)
        layers.append(output_layer)

        if final_activation.lower() != "identity":
            layers.append(FinalActivation())

        self.main_mlp = nn.Sequential(*layers)

    def forward(self, x, hidden_state=None):
        """
        Forward pass through the RNN + MLP over all time steps.
        Args:
            x: [B, T, input_dim]
            hidden_state: RNN hidden state.
        Returns:
            output: [B, T, output_dim] - predictions for each time step.
            hidden_state: updated hidden state.
        """
        if x.dim() == 2:
            x = x.unsqueeze(1)  # Ensure shape is [B, 1, input_dim]
        rnn_out, hidden_state = self.rnn_core(x, hidden_state)  # rnn_out: [B, T, H]
        
        # Instead of taking only the last time step:
        B, T, H = rnn_out.shape
        # Flatten the time dimension for a batched processing through the MLP head:
        rnn_out_flat = rnn_out.contiguous().view(B * T, H)
        output_flat = self.main_mlp(rnn_out_flat)  # shape: [B*T, output_dim]
        # Reshape back to a sequence:
        output = output_flat.view(B, T, -1)
        return output, hidden_state


    def init_hidden(self, batch_size=1, device='cpu'):
        """
        Initialize hidden state for GRU or LSTM.
        """
        num_layers = self.rnn_core.num_layers
        hidden_dim = self.hidden_dim
        if self.rnn_cell_type == 'lstm':
            # LSTM needs (h_0, c_0)
            h_0 = torch.zeros(num_layers, batch_size, hidden_dim, device=device)
            c_0 = torch.zeros(num_layers, batch_size, hidden_dim, device=device)
            return (h_0, c_0)
        else:
            # GRU needs only h_0
            h_0 = torch.zeros(num_layers, batch_size, hidden_dim, device=device)
            return h_0


In [ ]:
#| export
class BaseApproximatorRNN(BaseApproximator, nn.Module):

    """ Some basic functions for approximators """

    def __init__(self):
        super().__init__()

    def build_RNN(  self,
                    input_size: int | List[int], # is List, it means that multiple inputs are used. The first element is alwas for the RNN, the rest for the MLP
                    output_size: int,
                    hidden_layers_RNN:int,
                    num_hidden_units_RNN: int,
                    hidden_layers_MLP:List,
                    hidden_layers_input_MLP: List | None = None, # If a separate MLP is used for (potential) MLP input
                    RNN_cell: str = "GRU",
                    activation: str = "relu",
                    drop_prob: float = 0.0,
                    batch_norm: bool = False,
                    final_activation: str = "identity",
                    init_method: str = "xavier_uniform" # Parameter for initialization
                  ):

        """ Builds a recurrent neural network (RNN) """

        if isinstance(input_size, int):
            RNN_input_size = input_size
            MLP_input_size = None
        elif isinstance(input_size, list):

            if len(input_size) != 2:
                raise ValueError(f"Input size must be a list of length 2 (got {len(input_size)}) with elementes (RNN_input_size, MLP_input_size)")

            RNN_input_size = input_size[0]
            MLP_input_size = input_size[1]

        else:
            raise ValueError("Input size must be an integer or a list of integers")
    
        self.model = RNNMLPHybrid(  RNN_input_size,
                                    MLP_input_size,
                                    output_size,
                                    num_hidden_units_RNN,
                                    hidden_layers_RNN,
                                    hidden_layers_MLP,
                                    hidden_layers_input_MLP,
                                    RNN_cell,
                                    activation,
                                    final_activation,
                                    drop_prob,
                                    batch_norm,
                                    init_method,
                                    )


        

In [ ]:
#| export
class BaseApproximatorRL2RNN(BaseApproximator, nn.Module):
    """
    Base approximator class for RL² RNN-based models.
    """

    def __init__(self):
        super().__init__()

    def build_RL2RNN(self,
                     input_size: int,
                     output_size: int,
                     hidden_layers_RNN: int,
                     num_hidden_units_RNN: int,
                     hidden_layers_MLP: list,
                     RNN_cell: str = "GRU",
                     activation: str = "relu",  # <-- RELU instead of tanh
                     drop_prob: float = 0.0,
                     batch_norm: bool = False,
                     final_activation: str = "identity",
                     init_method: str = "xavier_uniform"):
        """
        Builds a simple RNN -> optional MLP -> output model for RL².
        """

        self.model = RL2RNN(
            RNN_input_size=input_size,
            output_size=output_size,
            hidden_layers_RNN=hidden_layers_RNN,
            num_hidden_units_RNN=num_hidden_units_RNN,
            hidden_layers_MLP=hidden_layers_MLP,
            RNN_cell=RNN_cell,
            activation=activation,
            final_activation=final_activation,
            drop_prob=drop_prob,
            batch_norm=batch_norm,
            init_method=init_method
        )


In [ ]:
#| export
class MLPStateAction(BaseApproximatorMLP):

    """Multilayer perceptron model for critic networks that take
    both states and actions as inputs to output the q-value"""

    def __init__(self,
                    input_shape: Tuple | List[Tuple], # number of features
                    output_shape: Tuple, # number of outputs/actions
                    hidden_layers: list, # list of number of neurons in each hidden layer
                    activation: str = "relu",
                    drop_prob: float = 0.0, # dropout probability
                    batch_norm: bool = False, # whether to apply batch normalization
                    final_activation: str = "identity", # whether to apply ReLU activation to the output
                    init_method: str = "xavier_uniform",  # Parameter for initialization
                    use_cuda: bool = False, # handled by mushroomRL, not used here
                    dropout: bool = False # legacy parameter to ensure compatibility, use drop_prob instead
                    ):

        super().__init__()

        # if input shape is list, then concatenate the elements
        if isinstance(input_shape, list):
            input_shape = (sum([shape[0] for shape in input_shape]),)
        
        self.build_MLP(    input_shape[0],
                            output_shape[0],
                            hidden_layers,
                            activation, 
                            drop_prob,
                            batch_norm,
                            final_activation,
                            init_method)

    def forward(self, state, action):


        state_action = torch.cat([state.float(), action.float()], dim=1)

        q = self.model(state_action)

        # TODO: check if squeeze is necessary
        # return q
        return torch.squeeze(q)

In [ ]:
#| export
class MLPState(BaseApproximatorMLP):

    """Multilayer perceptron model for critic networks that take
    both states and actions as inputs to output the q-value"""

    def __init__(self,
                    input_shape: Tuple, # number of features
                    output_shape: Tuple, # number of outputs/actions
                    hidden_layers: list, # list of number of neurons in each hidden layer
                    activation: str = "relu",
                    drop_prob: float = 0.0, # dropout probability
                    batch_norm: bool = False, # whether to apply batch normalization
                    final_activation: str = "identity", # whether to apply ReLU activation to the output
                    init_method: str = "xavier_uniform",  # Parameter for initialization
                    use_cuda: bool = False, # handled by mushroomRL, not used here
                    dropout: bool = False # legacy parameter to ensure compatibility, use drop_prob instead
                    ):

        super().__init__()
        
        self.build_MLP(    input_shape[0],
                            output_shape[0],
                            hidden_layers,
                            activation, 
                            drop_prob,
                            batch_norm,
                            final_activation,
                            init_method)

    def forward(self, state):

        state = state.float()

        q = self.model(state)

        # TODO: check if squeeze is necessary
        # return q.squeeze()
        return q

In [ ]:
#| export
class MLPActor(BaseApproximatorMLP):

    """Multilayer perceptron model for critic networks that take
    both states and actions as inputs to output the q-value"""

    def __init__(self,
                    input_shape: Tuple, # number of features
                    output_shape: Tuple, # number of outputs/actions
                    hidden_layers: list, # list of number of neurons in each hidden layer
                    activation: str = "relu",
                    drop_prob: float = 0.0, # dropout probability
                    batch_norm: bool = False, # whether to apply batch normalization
                    final_activation: str = "identity", # whether to apply ReLU activation to the output
                    init_method: str = "xavier_uniform",  # Parameter for initialization
                    use_cuda: bool = False,
                    dropout: bool = False, # legacy parameter to ensure compatibility, use drop_prob instead
                    **kwargs
                    ): 

        super().__init__()
        
        self.build_MLP(    input_shape[0],
                            output_shape[0],
                            hidden_layers,
                            activation, 
                            drop_prob,
                            batch_norm,
                            final_activation,
                            init_method)

    def forward(self, state):

        state = state.float()

        a = self.model(state)

        return a

In [ ]:
#| export
class RNNActor(BaseApproximatorRNN):

    """Multilayer perceptron model for critic networks that take
    both states and actions as inputs to output the q-value"""

    def __init__(self,
                    input_shape: List[Tuple], # input shape, must be exaclty as input shape into agent for mushroom_rl to work
                    output_shape: Tuple, # number of outputs/actions
                    hidden_layers_RNN: int, # number of initial hidden RNN layers
                    num_hidden_units_RNN: int, # number of neurons in the RNN layers
                    hidden_layers_MLP: List, # list of number of neurons in each hidden MLP layer, following the RNN layers
                    hidden_layers_input_MLP: List | None = None, # If a separate MLP is used for (potential) MLP input
                    RNN_cell: str = "GRU", # RNN cell type
                    activation: str = "relu",
                    drop_prob: float = 0.0, # dropout probability
                    batch_norm: bool = False, # whether to apply batch normalization
                    final_activation: str = "identity", # whether to apply ReLU activation to the output
                    init_method: str = "xavier_uniform",  # Parameter for initialization
                    use_cuda: bool = False,
                    dropout: bool = False, # legacy parameter to ensure compatibility, use drop_prob instead
                    input_shape_: List[Tuple] = None, # input shape for composite spaces
                    **kwargs
                    ): 

        super().__init__()

        if input_shape_ is not None:
            input_shape = input_shape_

        if isinstance(input_shape, tuple):
            if len(input_shape) != 2:
                raise ValueError(f"Input shape must be a tuple with dimensions (time_steps, features), got {input_shape}")
            input_size = input_shape[1]
            self.rnn_shape = input_shape
            self.mlp_shape = None
            self.get_time_input = True
        else:
            if len(input_shape) > 2:
                raise ValueError(f"Input shape must be a tuple or a list of tuples with length 1 or 2, got length {len(input_shape)}")
            if len(input_shape) == 2:
                input_size = [input_shape[0][1], input_shape[1][0]]
                self.rnn_shape = input_shape[0]
                self.mlp_shape = input_shape[1]
            else:
                input_size = input_shape[0][1]
                self.rnn_shape = input_shape[0]
                self.mlp_shape = None
            self.get_time_input = False

        self.build_RNN(     input_size,
                            output_shape[0],
                            hidden_layers_RNN,
                            num_hidden_units_RNN,
                            hidden_layers_MLP,
                            hidden_layers_input_MLP,
                            RNN_cell,
                            activation, 
                            drop_prob,
                            batch_norm,
                            final_activation,
                            init_method)

    def forward(self, state):
        
        if self.get_time_input: # already in the right 2d (or 3d with batch) format
            a = self.model(state.float(), None)
        
        else:
            if state.dim() == 1:
                a = self.forward_without_batch(state.float())
            else:
                a = self.forward_with_batch(state.float())
                
        return a

    def forward_with_batch(self, state):
        if self.mlp_shape is not None:
            rnn_input = state[:, :self.rnn_shape[0] * self.rnn_shape[1]]
            mlp_input = state[:, self.rnn_shape[0] * self.rnn_shape[1]:]
        else:
            rnn_input = state
            mlp_input = None
    
        # Reshape rnn_input to (batch_size, time, features)
        rnn_input = rnn_input.view(-1, self.rnn_shape[0], self.rnn_shape[1])
        
        return self.model(rnn_input, mlp_input)

    def forward_without_batch(self, state):
        if self.mlp_shape is not None:
            rnn_input = state[:self.rnn_shape[0] * self.rnn_shape[1]]
            mlp_input = state[self.rnn_shape[0] * self.rnn_shape[1]:]
        else:
            rnn_input = state
            mlp_input = None
        
        # Reshape rnn_input to (time, features)
        rnn_input = rnn_input.view(self.rnn_shape[0], self.rnn_shape[1])
        
        return self.model(rnn_input, mlp_input)

In [ ]:
#| export
class RNNStateAction(BaseApproximatorRNN):

    """Multilayer perceptron model for critic networks that take
    both states and actions as inputs to output the q-value"""

    def __init__(self,
                    input_shape: List[Tuple], # input shape, must be exaclty as input shape into agent for mushroom_rl to work
                    output_shape: Tuple, # Output shape
                    hidden_layers_RNN: int, # number of initial hidden RNN layers
                    num_hidden_units_RNN: int, # number of neurons in the RNN layers
                    hidden_layers_MLP: List, # list of number of neurons in each hidden MLP layer, following the RNN layers
                    hidden_layers_input_MLP: List | None = None, # structure of MLP to speratly process non-RNN input
                    RNN_cell: str = "GRU", # RNN cell type
                    activation: str = "relu",
                    drop_prob: float = 0.0, # dropout probability
                    batch_norm: bool = False, # whether to apply batch normalization
                    final_activation: str = "identity", # whether to apply ReLU activation to the output
                    init_method: str = "xavier_uniform",  # Parameter for initialization
                    use_cuda: bool = False,
                    dropout: bool = False, # legacy parameter to ensure compatibility, use drop_prob instead
                    input_shape_: List[Tuple] = None, # input shape for composite spaces
                    **kwargs
                    ): 

        super().__init__()

        if input_shape_ is not None:
            input_shape = input_shape_

        # check that input lenght of list is 2 or 3
        if len(input_shape) != 2:
            raise ValueError(f"Input shape must be a list of length 2, got {len(input_shape)}")
        
        action_input = input_shape[1][0]
    
        if isinstance(input_shape[0], tuple):
            if len(input_shape) != 2:
                raise ValueError(f"Input shape must be a tuple with dimensions (time_steps, features), got {input_shape}")
            self.rnn_shape = input_shape[0]
            non_time_features = 0
            self.get_time_input = True
        else:
            if isinstance(input_shape[0], list):
                if len(input_shape[0]) > 2:
                    raise ValueError(f"Input shape must be a list of length 1 or 2, got {len(input_shape[0])}")
                self.rnn_shape = input_shape[0][0]
                if len(input_shape[0]) == 2:
                    non_time_features = input_shape[0][1][0]
                else:
                    non_time_features = 0
                self.get_time_input = False
            else:
                raise ValueError("Input shape for composite spaces must be a list")
        
        self.mlp_shape = (action_input + non_time_features,)

        self.build_RNN(     [self.rnn_shape[1], self.mlp_shape[0]],
                            output_shape[0],
                            hidden_layers_RNN,
                            num_hidden_units_RNN,
                            hidden_layers_MLP,
                            hidden_layers_input_MLP,
                            RNN_cell,
                            activation, 
                            drop_prob,
                            batch_norm,
                            final_activation,
                            init_method)

    def forward(self, state, action):

        if self.get_time_input: # already in the right 2d (or 3d with batch) format
            q = self.model(state.float(), action.float())
        
        else:
            if state.dim() == 1:
                q = self.forward_without_batch(state.float(), action.float())
            else:
                q = self.forward_with_batch(state.float(), action.float())

        return torch.squeeze(q)
    
    def forward_with_batch(self, state, action):
        
        if self.mlp_shape is not None:
            rnn_input = state[:, :self.rnn_shape[0] * self.rnn_shape[1]]
            mlp_input = state[:, self.rnn_shape[0] * self.rnn_shape[1]:]
        else:
            rnn_input = state
            mlp_input = None
    
        # Reshape rnn_input to (batch_size, time, features)
        rnn_input = rnn_input.view(-1, self.rnn_shape[0], self.rnn_shape[1])

        mlp_input = torch.cat((mlp_input, action), dim=1) # dim 0 is batch dimension
        
        return self.model(rnn_input, mlp_input)
    
    def forward_without_batch(self, state, action):
        if self.mlp_shape is not None:
            rnn_input = state[:self.rnn_shape[0] * self.rnn_shape[1]]
            mlp_input = state[self.rnn_shape[0] * self.rnn_shape[1:]:]
        else:
            rnn_input = state
            mlp_input = None
        
        # Reshape rnn_input to (time, features)
        rnn_input = rnn_input.view(self.rnn_shape[0], self.rnn_shape[1])

        mlp_input = torch.cat((mlp_input, action), dim=0) # no batch dimension  
        
        return self.model(rnn_input, mlp_input)

In [ ]:
#| export
class RL2RNNActor(BaseApproximatorRL2RNN):
    """
    RL² Actor network for continuous actions.
    Outputs action means directly (no std or log_std here).
    For LSTM: the hidden state is stored and returned as a single tensor of shape
    (2, num_layers, batch, hidden_dim) obtained by stacking h and c along a new dimension.
    For GRU: the hidden state is handled as a single tensor.
    """

    def __init__(self,
                 input_shape: tuple,
                 output_shape: tuple,
                 hidden_layers_RNN: int,
                 num_hidden_units_RNN: int,
                 hidden_layers_MLP: list,
                 RNN_cell: str = "GRU",
                 activation: str = "relu",
                 final_activation: str = "identity",
                 drop_prob: float = 0.0,
                 batch_norm: bool = False,
                 init_method: str = "xavier_uniform",
                 **kwargs):
        super().__init__()

        input_size = input_shape[0]
        # Build RNN + MLP model
        self.build_RL2RNN(
            input_size=input_size,
            output_size=output_shape[0],  # output size = action_dim
            hidden_layers_RNN=hidden_layers_RNN,
            num_hidden_units_RNN=num_hidden_units_RNN,
            hidden_layers_MLP=hidden_layers_MLP,
            RNN_cell=RNN_cell,
            activation=activation,
            final_activation=final_activation,
            drop_prob=drop_prob,
            batch_norm=batch_norm,
            init_method=init_method
        )

    def forward(self, state, hidden_state=None):
        """
        Forward pass.
        For LSTM:
          - If hidden_state is None, initialize it from state.
          - If a hidden_state is provided as a stacked tensor, split it into (h, c).
          - Pass (h, c) to the model.
          - Receive the new hidden state tuple (h_new, c_new) and stack it along a new dimension
            so that it becomes a tensor of shape (2, num_layers, batch, hidden_dim).
        For GRU:
          - If hidden_state is None, initialize it.
          - Use the provided hidden state directly.
        Returns:
            - mean: the actor’s output (action mean).
            - hidden_state_new: the new hidden state in the standardized format.
        """
        if self.model.rnn_cell_type == 'lstm':
            # If hidden_state is None, initialize it based on state's batch size.
            if hidden_state is None:
                batch_size = state.shape[0] if state.dim() > 0 else 1
                hidden_state = self.init_hidden(batch_size, device=state.device)
            # Extract h and c from the hidden_state.
            h, c = hidden_state[0], hidden_state[1]
            # Pass the tuple (h, c) to the model.
            mean, hidden_state_new = self.model(state, (h, c))
            h_new, c_new = hidden_state_new
            # Stack h_new and c_new along a new dimension.
            hidden_state_new = torch.stack((h_new, c_new), dim=0)
        else:
            # For GRU, if hidden_state is None, initialize it.
            if hidden_state is None:
                batch_size = state.shape[0] if state.dim() > 0 else 1
                hidden_state = self.init_hidden(batch_size, device=state.device)
            mean, hidden_state_new = self.model(state, hidden_state)

        return mean, hidden_state_new

    def init_hidden(self, batch_size=1, device='cpu'):
        """
        Initializes the hidden state.
        For LSTM: returns a single tensor of shape (2, num_layers, batch, hidden_dim)
                 by stacking h_0 and c_0 along a new dimension.
        For GRU: returns a tensor of shape (num_layers, batch, hidden_dim).
        """
        num_layers = self.model.rnn_core.rnn.num_layers
        hidden_dim = self.model.hidden_dim
        shape = (num_layers, batch_size, hidden_dim)

        if self.model.rnn_cell_type == 'lstm':
            h_0 = torch.zeros(shape, device=device)
            c_0 = torch.zeros(shape, device=device)
            # Stack h_0 and c_0 along a new outer dimension (resulting shape: (2, num_layers, batch, hidden_dim))
            return torch.stack((h_0, c_0), dim=0)
        else:
            return torch.zeros(shape, device=device)


In [ ]:
#| export
class RL2RNNValue(BaseApproximatorRL2RNN):
    """
    RL² Value network for predicting state values.
    Outputs a single scalar value.
    """

    def __init__(self,
                 input_shape: tuple,
                 output_shape: tuple,
                 hidden_layers_RNN: int,
                 num_hidden_units_RNN: int,
                 hidden_layers_MLP: list,
                 RNN_cell: str = "GRU",
                 activation: str = "relu",
                 final_activation: str = "identity",
                 drop_prob: float = 0.0,
                 batch_norm: bool = False,
                 init_method: str = "xavier_uniform",
                 **kwargs):
        super().__init__()

        input_size = input_shape[0]

        # Build RNN + MLP model
        self.build_RL2RNN(
            input_size=input_size,
            output_size=output_shape[0],  # output size = 1
            hidden_layers_RNN=hidden_layers_RNN,
            num_hidden_units_RNN=num_hidden_units_RNN,
            hidden_layers_MLP=hidden_layers_MLP,
            RNN_cell=RNN_cell,
            activation=activation,
            final_activation=final_activation,
            drop_prob=drop_prob,
            batch_norm=batch_norm,
            init_method=init_method
        )

    def forward(self, state, hidden_state=None):
        """
        Forward pass.
        For LSTM:
          - If hidden_state is None, initialize it.
          - Split the hidden_state into (h, c), pass them to the model,
            then stack the output hidden state (h_new and c_new) along a new dimension.
        For GRU:
          - If hidden_state is None, initialize it.
          - Use the provided hidden state directly.
        """
        if self.model.rnn_cell_type == 'lstm':
            # If no hidden_state is provided, initialize it.
            if hidden_state is None:
                batch_size = state.shape[0] if state.dim() > 0 else 1
                hidden_state = self.init_hidden(batch_size, device=state.device)
            # Now extract h and c from the hidden_state.
            # We assume here that hidden_state is a stacked tensor with shape (2, num_layers, batch, hidden_dim)
            h, c = hidden_state[0], hidden_state[1]
            # Pass the (h, c) tuple to the model.
            mean, hidden_state_new = self.model(state, (h, c))
            h_new, c_new = hidden_state_new
            # Stack h_new and c_new along a new dimension so that the shape is (2, num_layers, batch, hidden_dim)
            hidden_state_new = torch.stack((h_new, c_new), dim=0)
        else:
            # For GRU, also initialize if hidden_state is None.
            if hidden_state is None:
                batch_size = state.shape[0] if state.dim() > 0 else 1
                hidden_state = self.init_hidden(batch_size, device=state.device)
            mean, hidden_state_new = self.model(state, hidden_state)

        return mean, hidden_state_new

    def init_hidden(self, batch_size=1, device='cpu'):
        """
        Initializes the hidden state.
        For LSTM: returns a single tensor of shape (2, num_layers, batch, hidden_dim)
                 by stacking h_0 and c_0.
        For GRU: returns a tensor of shape (num_layers, batch, hidden_dim).
        """
        num_layers = self.model.rnn_core.rnn.num_layers
        hidden_dim = self.model.hidden_dim
        shape = (num_layers, batch_size, hidden_dim)
        
        if self.model.rnn_cell_type == 'lstm':
            h_0 = torch.zeros(shape, device=device)
            c_0 = torch.zeros(shape, device=device)
            return torch.stack((h_0, c_0), dim=0)
        else:
            return torch.zeros(shape, device=device)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
import mushroom_rl
mushroom_rl.__file__

'/Users/magnus/miniforge3/envs/inventory_gym_2/lib/python3.11/site-packages/mushroom_rl/__init__.py'